# Проверка метрик: Baseline vs Calibrated vs Enriched GraphRAG

In [ ]:
import os
import sys
from pathlib import Path
import json
import shutil
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

env_path = PROJECT_ROOT / '.env'
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ[k.strip()] = v.strip().strip('"').strip("'")

DATA_DIR = Path('data/multihop_rag')
INDEX_DIR = Path('work/multihop_graphrag_openrouter')
ENRICHED_DIR = Path('work/multihop_graphrag_enriched')
LIMIT_DOCS = 50
LIMIT_QA = 25
MAX_LLM_UNITS = None
TOP_K = 10
RETRIEVAL_MODE = 'two_stage'

# Лимит рёбер/сущностей для обогащения (None = все)
MAX_ENRICH_EDGES = None
MAX_ENRICH_ENTITIES = None

# Шаги обогащения: 'questions', 'summaries', 'contradictions', 'entities', 'infer', 'importance'
ENRICH_STEPS = ['questions', 'summaries', 'contradictions', 'entities', 'infer', 'importance']

# Пропустить индексацию/загрузку датасета если индекс уже построен
SKIP_IF_INDEX_EXISTS = True

import importlib
import ecgraphrag.openrouter as openrouter_module
importlib.invalidate_caches()
importlib.reload(openrouter_module)
OpenRouterConfig = openrouter_module.OpenRouterConfig
openrouter_config = OpenRouterConfig.from_env()
print(f'Project root: {PROJECT_ROOT}')
print(f'OPENROUTER_API_KEY set: {bool(openrouter_config.api_key)}')
print(f'OpenRouter model: {openrouter_config.model}')
print(f'OpenRouter max_tokens: {openrouter_config.max_tokens}')
print(f'OpenRouter timeout: {openrouter_config.timeout}')
print(f'OpenRouter retries: {openrouter_config.retries}')
print('После изменения .env повторно выполните эту ячейку или перезапустите kernel.')

## 1. Загрузка MultiHop-RAG

Датасет берётся из репозитория `yixuantt/MultiHop-RAG`. Ячейка сохраняет нормализованные файлы `documents.jsonl` и `qa.jsonl`.

In [ ]:
if SKIP_IF_INDEX_EXISTS and (INDEX_DIR / 'calibrated_edges.jsonl').exists():
    print(f'Индекс уже существует в {INDEX_DIR}, пропускаем загрузку датасета.')
    manifest = {'status': 'skipped', 'reason': 'index already exists'}
else:
    from ecgraphrag.dataset import download_multihop_rag_dataset
    manifest = download_multihop_rag_dataset(DATA_DIR, limit_docs=LIMIT_DOCS, limit_qa=LIMIT_QA)
manifest

## 2. Индексация GraphRAG через OpenRouter LLM

In [ ]:
if SKIP_IF_INDEX_EXISTS and (INDEX_DIR / 'calibrated_edges.jsonl').exists():
    from ecgraphrag.storage import read_jsonl
    counts = {'status': 'skipped (index exists)', 'edges': len(read_jsonl(INDEX_DIR / 'calibrated_edges.jsonl'))}
    print(f'Индексация пропущена. Рёбер в индексе: {counts["edges"]}')
else:
    import importlib
    import ecgraphrag.openrouter as openrouter_module
    import ecgraphrag.extract as extract_module
    import ecgraphrag.indexer as indexer_module
    importlib.invalidate_caches()
    importlib.reload(openrouter_module)
    importlib.reload(extract_module)
    importlib.reload(indexer_module)
    indexer = indexer_module.GraphRAGIndexer(
        chunk_size=600,
        overlap=100,
        extractor='llm',
        max_llm_units=MAX_LLM_UNITS,
        resume=True,
    )
    counts = indexer.index(DATA_DIR / 'documents.jsonl', INDEX_DIR)
print({k: counts.get(k, 0) for k in ['successful_units', 'cached_units', 'failed_units']})
counts

## 3. LLM-обогащение графа

На этом шаге LLM добавляет к рёбрам и нодам семантическую информацию:

- **Questions** — 3-5 вопросов, на которые отвечает каждое ребро (vocabulary gap bridge)
- **Summaries** — fluent rephrasing механического `edge_text` для лучшего matching
- **Contradictions** — семантический анализ конфликтов между рёбрами одной пары сущностей
- **Entity enrichment** — rich-описания, aliases, категории для нод
- **Inferred edges** — новые рёбра из 2-hop транзитивных цепочек
- **Importance** — рейтинг критичности факта (0.0–1.0)

In [ ]:
from dataclasses import asdict
from ecgraphrag.enrich import enrich_graph, finalize_enriched_index
from ecgraphrag.models import Edge, Entity
from ecgraphrag.openrouter import OpenRouterClient
from ecgraphrag.storage import read_jsonl, export_table

raw_edges = read_jsonl(INDEX_DIR / 'calibrated_edges.jsonl')
raw_entities = read_jsonl(INDEX_DIR / 'entities.jsonl')

all_edges = [Edge(**row) for row in raw_edges]
all_entities = [Entity(**row) for row in raw_entities]

edges_to_enrich = all_edges[:MAX_ENRICH_EDGES] if MAX_ENRICH_EDGES else all_edges
entities_to_enrich = all_entities[:MAX_ENRICH_ENTITIES] if MAX_ENRICH_ENTITIES else all_entities

print(f'Всего: {len(all_edges)} рёбер, {len(all_entities)} сущностей')
print(f'Обогащаем: {len(edges_to_enrich)} рёбер, {len(entities_to_enrich)} сущностей')
print(f'Шаги: {ENRICH_STEPS}')

In [ ]:
# True — загрузить ранее сохранённый обогащённый граф
# False — повторно запустить LLM-обогащение
SKIP_IF_ENRICHED_EXISTS = True

enriched_edges_path = ENRICHED_DIR / 'calibrated_edges.jsonl'
enriched_entities_path = ENRICHED_DIR / 'entities.jsonl'

if (
    SKIP_IF_ENRICHED_EXISTS
    and enriched_edges_path.exists()
    and enriched_entities_path.exists()
):
    enriched_edges = [
        Edge(**row) for row in read_jsonl(enriched_edges_path)
    ]
    enriched_entities = [
        Entity(**row) for row in read_jsonl(enriched_entities_path)
    ]

    new_inferred = sum(
        edge.evidence_type == 'inferred'
        for edge in enriched_edges
    )

    print(f'LLM-обогащение пропущено. Граф загружен из: {ENRICHED_DIR}')
else:
    client = OpenRouterClient()

    enriched_entities_subset, enriched_edges_subset = enrich_graph(
        entities_to_enrich,
        edges_to_enrich,
        client,
        steps=ENRICH_STEPS,
    )

    enriched_edge_ids = {edge.id for edge in enriched_edges_subset}
    remaining_edges = [
        edge for edge in all_edges
        if edge.id not in enriched_edge_ids
    ]
    enriched_edges = enriched_edges_subset + remaining_edges

    enriched_entity_ids = {entity.id for entity in enriched_entities_subset}
    remaining_entities = [
        entity for entity in all_entities
        if entity.id not in enriched_entity_ids
    ]
    enriched_entities = enriched_entities_subset + remaining_entities

    new_inferred = len(enriched_edges_subset) - len(edges_to_enrich)

print(f'После обогащения: {len(enriched_edges)} рёбер (было {len(all_edges)})')
print(f'Сущностей: {len(enriched_entities)}')
print(f'Новых inferred-рёбер: {new_inferred}')

In [ ]:
enriched_counts = finalize_enriched_index(
    INDEX_DIR, ENRICHED_DIR, enriched_entities, enriched_edges
)

print(f'Обогащённый индекс сохранён: {ENRICHED_DIR}')

## 4. Baseline vs Calibrated vs Enriched+Calibrated

- **Baseline** — `score = Utility(e,q)`, без калибровки
- **Calibrated** — `score = Reliability(e) * Utility(e,q)`
- **Enriched+Calibrated** — добавляет отдельные retrieval-поля для generated questions, semantic summaries и enriched entities
- **Retrieval mode** — `two_stage`: сначала независимо ранжирует документы, затем упаковывает snippets и добавляет graph context

In [ ]:
import importlib
import ecgraphrag.text as text_module
import ecgraphrag.retrieve as retrieve_module
import ecgraphrag.metrics as metrics_module
importlib.invalidate_caches()
importlib.reload(text_module)
importlib.reload(retrieve_module)
importlib.reload(metrics_module)
evaluate_retrieval = metrics_module.evaluate_retrieval
failed_evidence_units = metrics_module.failed_evidence_units

qa_path = DATA_DIR / 'qa.jsonl'
failed_gold_units = failed_evidence_units(INDEX_DIR, qa_path)

if failed_gold_units:
    print(f'Предупреждение: {len(failed_gold_units)} failed chunks относятся к gold evidence.')
    print('Evaluation продолжится, но метрики могут быть занижены из-за отсутствующих фактов.')
    display(pd.DataFrame(failed_gold_units))

baseline = evaluate_retrieval(
    INDEX_DIR, qa_path, top_k=TOP_K, mode=RETRIEVAL_MODE,
    calibrated=False, limit=LIMIT_QA,
)
calibrated = evaluate_retrieval(
    INDEX_DIR, qa_path, top_k=TOP_K, mode=RETRIEVAL_MODE,
    calibrated=True, limit=LIMIT_QA,
)
enriched = evaluate_retrieval(
    ENRICHED_DIR, qa_path, top_k=TOP_K, mode=RETRIEVAL_MODE,
    calibrated=True, limit=LIMIT_QA,
)

print('Evaluation complete.')

In [ ]:
from IPython.display import display

# Сводная таблица метрик
metrics_cols = ['all_evidence_success_at_k', 'recall_at_k', 'precision_at_k', 'mrr', 'ndcg_at_k', 'packed_context_recall_at_k', 'answer_hit_rate']
summary_cols = metrics_cols + ['count', 'total_qa', 'skipped_null_queries', 'skipped_incomplete_evidence']

summary = pd.DataFrame([
    {k: baseline[k] for k in summary_cols},
    {k: calibrated[k] for k in summary_cols},
    {k: enriched[k] for k in summary_cols},
])
summary.index = ['baseline', 'calibrated', 'enriched+calibrated']
display(summary)
pd.DataFrame(enriched['by_question_type']).T

In [ ]:
# Дельты: прирост enriched над baseline и calibrated
delta_vs_baseline = {k: round(enriched[k] - baseline[k], 6) for k in metrics_cols}
delta_vs_calibrated = {k: round(enriched[k] - calibrated[k], 6) for k in metrics_cols}

deltas = pd.DataFrame([delta_vs_baseline, delta_vs_calibrated])
deltas.index = ['enriched_minus_baseline', 'enriched_minus_calibrated']
deltas

## 5. Просмотр обогащённых рёбер

In [ ]:
enriched_df = pd.read_json(ENRICHED_DIR / 'calibrated_edges.jsonl', lines=True)

enrich_cols = ['source', 'relation', 'target', 'reliability', 'importance',
               'semantic_summary', 'generated_questions', 'contradiction_info']
available_cols = [c for c in enrich_cols if c in enriched_df.columns]
enriched_df[available_cols].sort_values('importance', ascending=False).head(15)

In [ ]:
n = len(enriched_df)
stats = {
    'Всего рёбер': n,
    'С semantic_summary': int((enriched_df['semantic_summary'].astype(str) != '').sum()) if 'semantic_summary' in enriched_df.columns else 0,
    'С generated_questions': int((enriched_df['generated_questions'].apply(lambda x: len(x) if isinstance(x, list) else 0) > 0).sum()) if 'generated_questions' in enriched_df.columns else 0,
    'С contradiction_info': int((enriched_df['contradiction_info'].astype(str) != '').sum()) if 'contradiction_info' in enriched_df.columns else 0,
    'Inferred-рёбра': int((enriched_df['evidence_type'] == 'inferred').sum()) if 'evidence_type' in enriched_df.columns else 0,
    'Importance > 0.7': int((enriched_df['importance'] > 0.7).sum()) if 'importance' in enriched_df.columns else 0,
    'Importance < 0.3': int((enriched_df['importance'] < 0.3).sum()) if 'importance' in enriched_df.columns else 0,
}
pd.DataFrame([stats], index=['count']).T

## 6. Просмотр обогащённых сущностей

In [ ]:
entities_df = pd.read_json(ENRICHED_DIR / 'entities.jsonl', lines=True)

entity_cols = ['title', 'type', 'category', 'enriched_description', 'aliases']
available_entity_cols = [c for c in entity_cols if c in entities_df.columns]
entities_df[available_entity_cols].head(15)

## 7. Визуализация и сохранение графа

Сохраняются:

- `graph.graphml` — для Gephi/NetworkX;
- `enriched_graph.html` — интерактивная визуализация обогащённого графа, если установлен `pyvis`;
- `graph.png` — статичная картинка.

In [ ]:
import importlib
from pathlib import Path
from IPython.display import FileLink, HTML, display

import ecgraphrag.visualize as visualize_module

importlib.invalidate_caches()
importlib.reload(visualize_module)

if not (ENRICHED_DIR / 'calibrated_edges.jsonl').exists():
    raise FileNotFoundError(f'Обогащённый индекс не найден: {ENRICHED_DIR}')

graph_exports = visualize_module.export_graph(
    ENRICHED_DIR,
    ENRICHED_DIR / 'graph_exports',
    max_nodes=120,
)

enriched_html_path = Path(graph_exports['html']).resolve()

print(f'Enriched graph HTML: {enriched_html_path}')
display(FileLink(str(enriched_html_path)))
display(HTML(f'<iframe src="{enriched_html_path.as_uri()}" width="100%" height="800"></iframe>'))

graph_exports

## 8. Проверки проекта из notebook

Эта ячейка запускает unit tests из notebook. Её нужно выполнять после изменений кода перед экспериментами.


In [ ]:
import subprocess

unit_test_env = os.environ.copy()
unit_test_env['PYTHONPATH'] = str(PROJECT_ROOT / 'src')
unit_test_result = subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
    cwd=PROJECT_ROOT,
    env=unit_test_env,
    text=True,
    capture_output=True,
)
print(unit_test_result.stdout)
print(unit_test_result.stderr)
if unit_test_result.returncode != 0:
    raise RuntimeError('Unit tests failed')
print('Unit tests passed.')


## 9. Конфигурация экспериментов для двух датасетов

Ablation выполняется на одном full enriched index для каждого датасета. Повторные LLM-вызовы не запускаются, если enriched index уже сохранён.


In [ ]:
EXPERIMENT_DATASETS = {
    'multihop_rag': {
        'data_dir': Path('data/multihop_rag'),
        'index_dir': Path('work/multihop_graphrag_openrouter'),
        'enriched_dir': Path('work/multihop_graphrag_enriched'),
        'limit_docs': LIMIT_DOCS,
        'limit_qa': LIMIT_QA,
        'max_llm_units': MAX_LLM_UNITS,
    },
    'musique_ans': {
        'data_dir': Path('data/musique_ans'),
        'index_dir': Path('work/musique_ans_graphrag_openrouter'),
        'enriched_dir': Path('work/musique_ans_graphrag_enriched'),
        'limit_docs': None,
        'limit_qa': LIMIT_QA,
        'max_llm_units': MAX_LLM_UNITS,
    },
}

RUN_DATASET_PREP = True
RUN_INDEXING = True
RUN_FULL_ENRICHMENT = True
SKIP_IF_DATASET_EXISTS = True
SKIP_IF_INDEX_EXISTS = True
SKIP_IF_ENRICHED_EXISTS = True

ABLATION_OUTPUT_ROOT = Path('work/ablations')
EXPERIMENT_DATASETS


## 10. Подготовка датасетов

Ячейка создаёт `documents.jsonl` и `qa.jsonl` для MultiHop-RAG и MuSiQue-Ans.


In [ ]:
from ecgraphrag.dataset import download_multihop_rag_dataset, download_musique_ans_dataset

prepared_datasets = {}
for dataset_name, cfg in EXPERIMENT_DATASETS.items():
    data_dir = cfg['data_dir']
    docs_path = data_dir / 'documents.jsonl'
    qa_path = data_dir / 'qa.jsonl'
    if SKIP_IF_DATASET_EXISTS and docs_path.exists() and qa_path.exists():
        manifest_path = data_dir / 'dataset_manifest.json'
        manifest = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.exists() else {'status': 'skipped'}
        print(f'{dataset_name}: dataset already exists in {data_dir}')
    elif not RUN_DATASET_PREP:
        raise RuntimeError(f'{dataset_name}: dataset files are missing and RUN_DATASET_PREP=False')
    elif dataset_name == 'multihop_rag':
        manifest = download_multihop_rag_dataset(
            data_dir,
            limit_docs=cfg['limit_docs'],
            limit_qa=cfg['limit_qa'],
        )
    elif dataset_name == 'musique_ans':
        manifest = download_musique_ans_dataset(
            data_dir,
            limit_qa=cfg['limit_qa'],
        )
    else:
        raise ValueError(dataset_name)
    prepared_datasets[dataset_name] = manifest

pd.DataFrame(prepared_datasets).T


## 11. Индексация baseline graph для каждого датасета

Индексация использует LLM extraction и resume cache. Если индекс уже есть, ячейка его переиспользует.


In [ ]:
from ecgraphrag.indexer import GraphRAGIndexer
from ecgraphrag.storage import read_jsonl

index_counts_by_dataset = {}
for dataset_name, cfg in EXPERIMENT_DATASETS.items():
    data_dir = cfg['data_dir']
    index_dir = cfg['index_dir']
    if SKIP_IF_INDEX_EXISTS and (index_dir / 'calibrated_edges.jsonl').exists():
        counts = {
            'status': 'skipped',
            'edges': len(read_jsonl(index_dir / 'calibrated_edges.jsonl')),
            'entities': len(read_jsonl(index_dir / 'entities.jsonl')),
        }
        print(f'{dataset_name}: index already exists in {index_dir}')
    elif not RUN_INDEXING:
        raise RuntimeError(f'{dataset_name}: index is missing and RUN_INDEXING=False')
    else:
        indexer = GraphRAGIndexer(
            chunk_size=600,
            overlap=100,
            extractor='llm',
            max_llm_units=cfg['max_llm_units'],
            resume=True,
        )
        counts = indexer.index(data_dir / 'documents.jsonl', index_dir)
    index_counts_by_dataset[dataset_name] = counts

pd.DataFrame(index_counts_by_dataset).T


## 12. Full enrichment для каждого датасета

Для каждого датасета создаётся один full enriched index со всеми enrichment steps. Ablation ниже будет включать и выключать enriched-сигналы только на retrieval side.


In [ ]:
from ecgraphrag.enrich import enrich_graph, finalize_enriched_index
from ecgraphrag.models import Edge, Entity
from ecgraphrag.openrouter import OpenRouterClient


def load_index_graph(index_dir: Path) -> tuple[list[Entity], list[Edge]]:
    entities = [Entity(**row) for row in read_jsonl(index_dir / 'entities.jsonl')]
    edges = [Edge(**row) for row in read_jsonl(index_dir / 'calibrated_edges.jsonl')]
    return entities, edges


def ensure_full_enriched_index(dataset_name: str, cfg: dict) -> dict:
    index_dir = cfg['index_dir']
    enriched_dir = cfg['enriched_dir']
    enriched_edges_path = enriched_dir / 'calibrated_edges.jsonl'
    enriched_entities_path = enriched_dir / 'entities.jsonl'
    if SKIP_IF_ENRICHED_EXISTS and enriched_edges_path.exists() and enriched_entities_path.exists():
        print(f'{dataset_name}: enriched index already exists in {enriched_dir}')
        return {
            'status': 'skipped',
            'edges': len(read_jsonl(enriched_edges_path)),
            'entities': len(read_jsonl(enriched_entities_path)),
            'inferred_edges': sum(
                row.get('evidence_type') == 'inferred'
                for row in read_jsonl(enriched_edges_path)
            ),
        }
    if not RUN_FULL_ENRICHMENT:
        raise RuntimeError(f'{dataset_name}: enriched index is missing and RUN_FULL_ENRICHMENT=False')

    all_entities, all_edges = load_index_graph(index_dir)
    edges_to_enrich = all_edges[:MAX_ENRICH_EDGES] if MAX_ENRICH_EDGES else all_edges
    entities_to_enrich = all_entities[:MAX_ENRICH_ENTITIES] if MAX_ENRICH_ENTITIES else all_entities
    print(f'{dataset_name}: enriching {len(edges_to_enrich)} edges and {len(entities_to_enrich)} entities')

    client = OpenRouterClient()
    enriched_entities_subset, enriched_edges_subset = enrich_graph(
        entities_to_enrich,
        edges_to_enrich,
        client,
        steps=ENRICH_STEPS,
    )

    enriched_edge_ids = {edge.id for edge in enriched_edges_subset}
    remaining_edges = [edge for edge in all_edges if edge.id not in enriched_edge_ids]
    enriched_edges = enriched_edges_subset + remaining_edges

    enriched_entity_ids = {entity.id for entity in enriched_entities_subset}
    remaining_entities = [entity for entity in all_entities if entity.id not in enriched_entity_ids]
    enriched_entities = enriched_entities_subset + remaining_entities

    counts = finalize_enriched_index(index_dir, enriched_dir, enriched_entities, enriched_edges)
    counts['inferred_edges'] = sum(edge.evidence_type == 'inferred' for edge in enriched_edges)
    return counts


enriched_counts_by_dataset = {}
for dataset_name, cfg in EXPERIMENT_DATASETS.items():
    enriched_counts_by_dataset[dataset_name] = ensure_full_enriched_index(dataset_name, cfg)

pd.DataFrame(enriched_counts_by_dataset).T


## 13. Retrieval-side ablation для каждого датасета

Ablation использует сохранённый full enriched index и измеряет вклад отдельных retrieval-сигналов: questions, summaries, entities, inferred edges, importance и contradiction penalty.


In [ ]:
import importlib
import ecgraphrag.benchmark as benchmark_module
importlib.invalidate_caches()
importlib.reload(benchmark_module)

run_retrieval_ablation = benchmark_module.run_retrieval_ablation
load_retrieval_config = benchmark_module.load_retrieval_config

ablation_results = {}
for dataset_name, cfg in EXPERIMENT_DATASETS.items():
    output_dir = ABLATION_OUTPUT_ROOT / dataset_name
    result = run_retrieval_ablation(
        dataset_name=dataset_name,
        data_path=cfg['data_dir'],
        index_path=cfg['index_dir'],
        enriched_index_path=cfg['enriched_dir'],
        output_path=output_dir,
        config=load_retrieval_config(None),
        top_k=TOP_K,
        limit=cfg['limit_qa'],
    )
    ablation_results[dataset_name] = result
    print(f'{dataset_name}: ablation saved to {output_dir}')

print('Ablation complete.')


## 14. Сводная таблица ablation

Таблица показывает метрики для каждого датасета и каждого retrieval-варианта. Поле `delta_*` считается относительно `calibrated_base_index` внутри того же датасета.


In [ ]:
ablation_rows = []
metric_names = [
    'all_evidence_success_at_k',
    'recall_at_k',
    'precision_at_k',
    'mrr',
    'ndcg_at_k',
    'packed_context_recall_at_k',
    'answer_hit_rate',
    'latency_ms_p50',
    'latency_ms_p95',
]

for dataset_name, result in ablation_results.items():
    for variant, summary_row in result['summary'].items():
        row = {
            'dataset': dataset_name,
            'variant': variant,
            'valid': summary_row.get('valid', True),
            'signals': '+'.join(summary_row.get('signals', [])),
        }
        for metric in metric_names:
            row[metric] = summary_row.get(metric, 0.0)
            row[f'delta_{metric}'] = summary_row.get('delta_vs_calibrated', {}).get(metric, 0.0)
        ablation_rows.append(row)

ablation_summary = pd.DataFrame(ablation_rows)
display(ablation_summary)

for dataset_name in ablation_results:
    csv_path = ABLATION_OUTPUT_ROOT / dataset_name / 'summary.csv'
    print(f'{dataset_name}: {csv_path}')
